# UCS761 - Deep Learning Lab 7
### From Neurons to Vision Models (CNN)

Everything we have built so far treats input as a flat list of numbers.

For images, that is a problem. A 28x28 image flattened into 784 numbers loses all spatial structure. A pixel in the top-left corner gets connected to a pixel in the bottom-right with the same weight style as adjacent pixels. That makes no sense.

CNN fixes this by using local filters that slide across the image. Same weights reused everywhere. That is the key idea.

This lab covers:
Why dense layers scale badly for images
What convolution actually is (with a manual example)
Filters, feature maps, padding, pooling, dropout
CNN on MNIST using Keras

In [9]:
import numpy as np

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    print("TensorFlow:", tf.__version__)
    TF = True
except ImportError:
    print("TensorFlow not installed. Will show concepts and architecture only.")
    TF = False


TensorFlow: 2.19.0


## Why Dense Layers Break for Images

A 28x28 grayscale image = 784 pixel values.

If the first hidden layer has 128 neurons:
Parameters = 784 x 128 + 128 = 100,480 (just the first layer)

A 3x3 CNN filter:
Parameters = 9 weights + 1 bias = 10
And those 10 weights get reused at every single position in the image.

Dense layers also treat every pixel as equally relevant to every other pixel. They connect a top-left pixel to a bottom-right pixel with separate weights. In reality nearby pixels are much more related than distant ones. Dense layers ignore that completely.

In [10]:
img_h, img_w = 28, 28
hidden1 = 128

dense_params = (img_h * img_w) * hidden1 + hidden1
conv_params  = 16 * (3*3 + 1)

print(f"Dense first layer params:  {dense_params:,}")
print(f"Conv layer (16 filters):   {conv_params}")
print(f"Ratio: {dense_params // conv_params}x fewer parameters with CNN")


Dense first layer params:  100,480
Conv layer (16 filters):   160
Ratio: 628x fewer parameters with CNN


## What Convolution Actually Does

A filter is a small grid of weights (say 3x3). It slides across the input.
At each position: multiply elementwise with the patch, then sum. That gives one output value.

Slide across the whole image and you get one feature map per filter.

The same 9 weights are reused at every position. This is parameter sharing. The model learns one edge detector and applies it everywhere, instead of learning a separate one for every pixel location.

In [11]:
def convolve2d(img, kernel):
    ih, iw = img.shape
    kh, kw = kernel.shape
    out_h  = ih - kh + 1
    out_w  = iw - kw + 1
    out    = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            out[i, j] = np.sum(img[i:i+kh, j:j+kw] * kernel)
    return out


horizontal_edge = np.array([
    [-1, -1, -1],
    [ 0,  0,  0],
    [ 1,  1,  1]
], dtype=float)

sample_img = np.array([
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1]
], dtype=float)

result = convolve2d(sample_img, horizontal_edge)
print("Input image:")
print(sample_img)
print("Filter (horizontal edge detector):")
print(horizontal_edge)
print("Convolution output:")
print(result)
print("Strong positive response where the horizontal edge is.")


Input image:
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]
Filter (horizontal edge detector):
[[-1. -1. -1.]
 [ 0.  0.  0.]
 [ 1.  1.  1.]]
Convolution output:
[[3. 3. 3.]
 [3. 3. 3.]
 [0. 0. 0.]]
Strong positive response where the horizontal edge is.


## Output Size Formula

When a filter is applied without padding, the output shrinks.

output_size = floor((N - F + 2P) / S) + 1

N: input size, F: filter size, P: padding, S: stride

Padding adds zeros around the border to keep spatial size from shrinking too fast.

In [12]:
def out_size(N, F, P=0, S=1):
    return (N - F + 2*P) // S + 1

print("Output size examples:")
print(f"  28x28, filter 3x3, no padding:   {out_size(28,3,0)} x {out_size(28,3,0)}")
print(f"  28x28, filter 3x3, padding=1:    {out_size(28,3,1)} x {out_size(28,3,1)}")
print(f"  After 2x2 max pool (stride 2):   {out_size(28,2,0,2)} x {out_size(28,2,0,2)}")


Output size examples:
  28x28, filter 3x3, no padding:   26 x 26
  28x28, filter 3x3, padding=1:    28 x 28
  After 2x2 max pool (stride 2):   14 x 14


## Max Pooling

After convolution we usually apply pooling: take the max value in each small region.

Purpose:
Reduces spatial dimensions (less computation downstream)
Gives some translation invariance (small shift in input, same output)
Keeps the most prominent response from each region

In [13]:
def maxpool2x2(fmap):
    h, w   = fmap.shape
    oh, ow = h//2, w//2
    out    = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = fmap[2*i:2*i+2, 2*j:2*j+2].max()
    return out

test_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [7, 8, 3, 0],
    [2, 1, 9, 4]
], dtype=float)

pooled = maxpool2x2(test_map)
print("Before pooling (4x4):")
print(test_map)
print("After 2x2 max pooling (2x2):")
print(pooled)


Before pooling (4x4):
[[1. 3. 2. 4.]
 [5. 6. 1. 2.]
 [7. 8. 3. 0.]
 [2. 1. 9. 4.]]
After 2x2 max pooling (2x2):
[[6. 4.]
 [8. 9.]]


## CNN on MNIST

Building and training a CNN on MNIST digit classification.

Architecture:
Conv2D(32, 3x3, relu, same) then MaxPool(2x2)
Conv2D(64, 3x3, relu, same) then MaxPool(2x2)
Flatten then Dense(128, relu) then Dropout(0.3) then Dense(10, softmax)

In [14]:
if TF:
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()

    x_tr = x_tr.astype("float32") / 255.0
    x_te = x_te.astype("float32") / 255.0

    x_tr = x_tr[..., np.newaxis]
    x_te = x_te[..., np.newaxis]

    print("Train:", x_tr.shape, " Test:", x_te.shape)
else:
    print("Dataset: 60,000 training + 10,000 test 28x28 grayscale digit images")
    print("Labels: 0 through 9")


Train: (60000, 28, 28, 1)  Test: (10000, 28, 28, 1)


In [15]:
if TF:
    cnn = keras.Sequential([
        layers.Conv2D(32, (3,3), activation="relu", padding="same", input_shape=(28,28,1)),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation="relu", padding="same"),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ])
    cnn.summary()
else:
    print("Architecture (if TF was available):")
    print("  Input:        (28, 28, 1)")
    print("  Conv(32,3x3) → (28, 28, 32)  [same padding]")
    print("  MaxPool(2x2) → (14, 14, 32)")
    print("  Conv(64,3x3) → (14, 14, 64)  [same padding]")
    print("  MaxPool(2x2) → ( 7,  7, 64)")
    print("  Flatten      → (3136,)")
    print("  Dense(128)   → (128,)")
    print("  Dropout(0.3)")
    print("  Dense(10)    → (10,)  [softmax]")


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,642 (1.61 MB)

 Trainable params: 421,642 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
if TF:
    cnn.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    hist = cnn.fit(x_tr, y_tr, epochs=5, batch_size=64, validation_split=0.1)

    loss, acc = cnn.evaluate(x_te, y_te, verbose=0)
    print(f"CNN Test Accuracy:  {acc*100:.2f}%")
    print(f"CNN Test Loss:      {loss:.4f}")


Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 59s 68ms/step - accuracy: 0.8549 - loss: 0.4533 - val_accuracy: 0.9847 - val_loss: 0.0531
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 57s 68ms/step - accuracy: 0.9790 - loss: 0.0717 - val_accuracy: 0.9872 - val_loss: 0.0448
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 83s 69ms/step - accuracy: 0.9852 - loss: 0.0503 - val_accuracy: 0.9880 - val_loss: 0.0462
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 58s 69ms/step - accuracy: 0.9880 - loss: 0.0385 - val_accuracy: 0.9892 - val_loss: 0.0391
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 57s 68ms/step - accuracy: 0.9906 - loss: 0.0310 - val_accuracy: 0.9917 - val_loss: 0.0311
CNN Test Accuracy:  99.14%
CNN Test Loss:      0.0263


## Dense Network Comparison

Training a plain fully connected network on the same data to compare results.

In [17]:
if TF:
    dense_net = keras.Sequential([
        layers.Flatten(input_shape=(28,28,1)),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    dense_net.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    dense_net.fit(x_tr, y_tr, epochs=5, batch_size=64, validation_split=0.1, verbose=1)

    d_loss, d_acc = dense_net.evaluate(x_te, y_te, verbose=0)

    print(f"CNN   Test Accuracy: {acc*100:.2f}%  |  Params: {cnn.count_params():,}")
    print(f"Dense Test Accuracy: {d_acc*100:.2f}%  |  Params: {dense_net.count_params():,}")
else:
    print("Expected results (typical):")
    print("  CNN   accuracy: ~99%     Parameters: ~93,000")
    print("  Dense accuracy: ~97-98%  Parameters: ~200,000+")
    print("CNN is both more accurate and uses fewer parameters.")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.8767 - loss: 0.4334 - val_accuracy: 0.9698 - val_loss: 0.1058
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9717 - loss: 0.0964 - val_accuracy: 0.9762 - val_loss: 0.0792
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9815 - loss: 0.0616 - val_accuracy: 0.9768 - val_loss: 0.0795
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9852 - loss: 0.0473 - val_accuracy: 0.9777 - val_loss: 0.0782
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9907 - loss: 0.0297 - val_accuracy: 0.9778 - val_loss: 0.0815
CNN   Test Accuracy: 99.14%  |  Params: 421,642
Dense Test Accuracy: 97.69%  |  Params: 235,146


## Answers to Key Questions

**Why does a dense layer scale badly for images:**
It treats every input as independent. Connects every pixel to every neuron with separate weights. Completely ignores the fact that nearby pixels are related. An edge in the top-left uses different weights than an edge in the bottom-right even though they are the same kind of feature.

**What is a filter and what is a feature map:**
A filter is a small grid of learned weights (e.g. 3x3 = 9 values). It acts as a pattern detector. When convolved with a region that matches its pattern, the response is high. A feature map is the output of sliding that filter across the entire input. Each filter produces one feature map.

**Why does activation still matter after convolution:**
Convolution is a linear operation (weighted sum). Stacking convolutions without activation would still be linear overall. ReLU after each conv layer adds the non-linearity needed to learn complex patterns.

**What does padding control:**
Without padding, each conv layer shrinks the spatial dimensions. For deep networks you would run out of resolution quickly. Same padding adds zeros around the border so input and output have the same height and width.

**Why is dropout useful:**
During training it randomly sets some neuron outputs to zero. Prevents the network from relying too heavily on specific neurons. Forces the model to learn redundant pathways to reach the right answer. At test time all neurons are used.

**Connection to DLQ6:**
DLQ6 showed that gradients can vanish in deep networks with sigmoid. CNNs have the same problem. That is why ReLU is standard in modern CNNs and why He initialization is preferred.